In [ ]:
import os,json
from typing import Sequence

from langchain_core.messages import message_to_dict,messages_from_dict,BaseMessage
from langchain_core.chat_history import BaseChatMessageHistory

# message_to_dict:单个消息对象（BaseMessage类实例）-> 字典
# message_from_dict:[字典，字典....] -> [消息，消息....]
# AIMessage,HumanMessage,SystemMessage都是BaseMessage的子类

class FileChatMessageHistory(BaseChatMessageHistory):
    def __init__(self,session_id,storage_path):
        self.session_id=session_id  #会话id
        self.storage_path=storage_path  #不同会话id的存储文件，所在的文件夹路径
        #完整的文件路径
        self.file_path=os.path.join(self.storage_path,self.session_id) #文件路径拼接

        #确保文件夹是否存在，如果不存在则自动创建
        os.makedirs(self.file_path,exist_ok=True)

    # 添加会话记忆
    # 参数后面的messages是新的消息
    def add_messages(self, messages: Sequence[BaseMessage]) -> None:
        # Sequence序列 类似list,tuple
        all_messages=list(self.messages)    # 已有的消息列表
        all_messages.extend(messages)   # 新的和已有的融合成一个list

        #将数据同步写入到本地文件中
        #类对象写入文件 -> 一堆二进制
        #为了方便，可以将BaseMessage消息转为字典（借助json模块对json字符串写入文件）
        #官方message_to_dict:单个消息对象（BaseMessage类实例） -> 字典

        new_messages=[message_to_dict(message) for message in all_messages]
        #将数据写入文件
        with open(self.file_path,'w',encoding='utf-8') as f:
            json.dump(new_messages,f)

    # 加载会话记忆
    @property   # @property装饰器将messages方法变成成员属性用
    def messages(self)->list[BaseMessage]:
        try:
            with open(self.file_path,'r',encoding='utf-8') as f:
                messages_data=json.load(f)   # 返回值就是：list[字典]
                #还得转换为消息
                return messages_from_dict(messages_data)
        except FileNotFoundError:
            return []

    # 清除会话记忆
    def clear(self)->None:
        with open(self.file_path,'w',encoding='utf-8') as f:
            json.dump([],f)